In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章只在 K=3~10 维向量上演示 softmax / 熵 / KL，CPU 一秒跑完。
# 有 GPU 也用不上 —— 把 .to('cuda') 这种语句省掉。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')


In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行，否则会被当成 line magic 报错。
# matplotlib：画概率柱状图、温度对分布的影响、KL 不对称的对比
# torch 在 Colab 已预装，这里不动。
!pip install -q -U matplotlib


In [ ]:
# ============================================================
# Cell 2: 从 logits 到概率 —— 手写 softmax & torch.softmax 验证一致
# ============================================================
# 关键概念：logits 是任意实数；softmax 把它归一成 ∑=1 的概率向量。
import torch
import torch.nn.functional as F

# 一条长度 K=3 的 logits（任意实数，可以负、可以大于 1）
logits = torch.tensor([2.0, 1.0, 0.1])

# 朴素 softmax（按定义直接算，logits 大时会数值溢出）
p_naive = torch.exp(logits) / torch.exp(logits).sum()

# 数值稳定 softmax（减去最大值，平移不变性保证结果一致）
m = logits.max()
p_stable = torch.exp(logits - m) / torch.exp(logits - m).sum()

# PyTorch 内置（推荐用这个，工业级稳定 + 自动求导友好）
p_torch = F.softmax(logits, dim=-1)

print('logits        =', logits.tolist())
print('softmax 朴素  =', [round(x, 4) for x in p_naive.tolist()])
print('softmax 稳定  =', [round(x, 4) for x in p_stable.tolist()])
print('softmax torch =', [round(x, 4) for x in p_torch.tolist()])
print('和   =', round(p_torch.sum().item(), 6), '   ← 必须是 1.0')
print('三种实现结果是否一致：', torch.allclose(p_naive, p_torch) and torch.allclose(p_stable, p_torch))


In [ ]:
# ============================================================
# Cell 3: 温度 T 怎么改变分布的尖锐 / 扁平
# ============================================================
# softmax(logits / T)：T>1 让分布更扁平（更随机）；T<1 让分布更尖锐（更确定）
# 这就是 LLM 推理里 temperature 参数背后的数学。
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

logits = torch.tensor([2.0, 1.0, 0.5, 0.0, -1.0])      # 5 类，差距明显
temperatures = [0.3, 0.7, 1.0, 1.5, 3.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 3), sharey=True)
for ax, T in zip(axes, temperatures):
    p = F.softmax(logits / T, dim=-1)                  # 注意：先除以 T 再 softmax
    ax.bar(range(len(logits)), p.tolist(), color='steelblue')
    ax.set_title(f'T = {T}')
    ax.set_ylim(0, 1)
    ax.set_xticks(range(len(logits)))
plt.suptitle('softmax(logits / T): higher T flattens the distribution')
plt.tight_layout()
plt.show()

# 量化"扁平程度"：算每个温度下分布的熵 H(p) = -Σ p_i log p_i（单位 nat，P03 第 4 节会正式讲）
# T 越大分布越扁平、熵越大；T → ∞ 时趋近上界 log K（均匀分布）
print('温度 T 与分布的熵：')
for T in temperatures:
    p = F.softmax(logits / T, dim=-1)
    H = -(p * torch.log(p)).sum().item()
    print(f'  T = {T:<4} →  H = {H:.4f} nat')
print(f'熵的上界 log K = log 5 = {math.log(5):.4f} nat')

# 顺手验证：T=∞ 极限是均匀分布（每类都是 1/K）
p_huge_T = F.softmax(logits / 1e6, dim=-1)
print('\nT → ∞ 时 softmax(logits/T) ≈ ', [round(x, 4) for x in p_huge_T.tolist()],
      '   ← 趋近均匀分布 1/K =', round(1 / len(logits), 4))

In [ ]:
# ============================================================
# Cell 4: 数值稳定的 log_softmax —— 不要"先 softmax 再取 log"
# ============================================================
# "先 softmax 再 log" 两头都会坏：
#   上溢：logits 很大时 exp() 直接溢出成 inf，inf/inf = nan，概率全是 nan；
#   下溢：logits 相差悬殊时小概率被压成 0，log(0) = -inf。
# 正解：直接用 F.log_softmax，它内部做 log-sum-exp，两头都稳定。
import torch
import torch.nn.functional as F

# 上溢示例：故意构造一组很大的 logits（实际训练里量级可能没这么夸张，但思路一样）
logits = torch.tensor([1000.0, 999.0, 998.0])

# ❌ 朴素：exp(1000) 在 float32 下直接 inf，inf/inf = nan，结果全 NaN
p_bad = torch.exp(logits) / torch.exp(logits).sum()
log_bad = torch.log(p_bad)
print('朴素 log(softmax) =', log_bad.tolist(), '  ← 全是 nan（exp 上溢成 inf，inf/inf = nan）')

# 下溢示例：logits 相差悬殊，小概率被舍入成 0，取 log 拿到 -inf
logits2 = torch.tensor([0.0, -200.0])
p_under = torch.exp(logits2) / torch.exp(logits2).sum()
print('下溢那头 log(softmax) =', torch.log(p_under).tolist(), '  ← 第二项 -inf（概率被压成 0）')

# ✅ PyTorch 的 log_softmax：内部减最大值 + log-sum-exp，两组 logits 都稳定
log_good = F.log_softmax(logits, dim=-1)
print('F.log_softmax（上溢组）=', [round(x, 4) for x in log_good.tolist()])
print('F.log_softmax（下溢组）=', [round(x, 4) for x in F.log_softmax(logits2, dim=-1).tolist()])

# 之所以稳定，本质上是用了恒等式：
#   log_softmax(ℓ)_i = ℓ_i - logsumexp(ℓ)
# 而 logsumexp 内部又会减最大值再加回来：
#   logsumexp(ℓ) = max + log Σ exp(ℓ - max)
print('logsumexp(logits) =', round(torch.logsumexp(logits, dim=-1).item(), 4))

In [ ]:
# ============================================================
# Cell 5: 熵 H(p) —— 一点全押 / 均匀 / 中间，三种典型分布
# ============================================================
# H(p) = -∑ p_i log p_i  ，单位 nat（PyTorch 用自然对数）
# 上界 = log K（均匀分布）；下界 = 0（一点全押）。
# 约定：0 · log 0 = 0。理由是 lim_{x→0+} x log x = 0（x 趋零比 log x 趋 -∞ 快），
# 直觉上「概率为 0 的事件不会发生，对不确定性贡献为 0」也对得上。
# 代码里加一个极小 eps 让数值实现回避 log(0) = -inf，与上面的约定结果一致。
import torch
import math

def entropy(p, eps=1e-12):
    p = torch.as_tensor(p, dtype=torch.float64)
    return -(p * torch.log(p + eps)).sum().item()       # 加 eps 落实 0·log0=0 的约定

K = 3
distributions = {
    '一点全押 (1, 0, 0)':           [1.0, 0.0, 0.0],
    '部分确定 (0.7, 0.2, 0.1)':     [0.7, 0.2, 0.1],
    '半半半 (0.5, 0.5, 0)':         [0.5, 0.5, 0.0],
    '均匀 (1/3, 1/3, 1/3)':         [1/3, 1/3, 1/3],
}

print(f'{"分布":35s} {"H (nat)":>10s} {"H (bit)":>10s}')
print('-' * 60)
for name, p in distributions.items():
    h_nat = entropy(p)
    h_bit = h_nat / math.log(2)                          # 换算到 bit
    print(f'{name:35s} {h_nat:10.4f} {h_bit:10.4f}')

print(f'\n理论上界 log K = log {K} = {math.log(K):.4f} nat = {math.log2(K):.4f} bit')
print('观察：均匀分布的熵恰好等于 log K，与公式一致。')


In [ ]:
# ============================================================
# Cell 6: 交叉熵 H(p, q) 与负对数似然 NLL 的等价性
# ============================================================
# 当 p 是 one-hot 时：H(p, q) = -log q[真实类别] = NLL
# PyTorch 的 F.cross_entropy 期望 input 是 raw logits、target 是类别 id。
import torch
import torch.nn.functional as F

# 模拟一个 mini-batch：N=4 个样本，K=5 类
torch.manual_seed(0)
logits = torch.randn(4, 5)                               # raw logits，任意实数
targets = torch.tensor([2, 0, 4, 1])                     # 每个样本的真实类别 id

# 方法 A：F.cross_entropy（一行就完事）
loss_a = F.cross_entropy(logits, targets, reduction='mean')

# 方法 B：手写 NLL = -log_softmax 在真实类别那位上取值再求平均
log_probs = F.log_softmax(logits, dim=-1)               # (N, K)
# 用 gather 把每行真实类别那一位的 log p 取出来，shape (N,)
log_p_true = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
loss_b = -log_p_true.mean()

# 方法 C：手写交叉熵 H(p, q)，p 是 one-hot
p_onehot = F.one_hot(targets, num_classes=5).float()    # (N, K)
loss_c = -(p_onehot * log_probs).sum(dim=-1).mean()

print(f'A. F.cross_entropy            = {loss_a.item():.6f}')
print(f'B. 手写 NLL (gather log_p)    = {loss_b.item():.6f}')
print(f'C. 手写 H(p, q), p one-hot    = {loss_c.item():.6f}')
print(f'三种实现完全一致：{torch.allclose(loss_a, loss_b) and torch.allclose(loss_a, loss_c)}')
print('\n结论：F.cross_entropy ≡ 手写 NLL ≡ one-hot 标签下的交叉熵。')


In [ ]:
# ============================================================
# Cell 7: 反例 —— 把 softmax 喂给 cross_entropy 会出什么错
# ============================================================
# 这是 PyTorch 用户最常踩的坑：cross_entropy 会再做一次 log_softmax。
# 错的版本不会报错，而且非常隐蔽：小任务上准确率往往照常上升（双重
# softmax 是单调变换，分类边界还能学出来），训练曲线看不出异常——
# 真正坏掉的是 loss 的含义（不再是 NLL、有一个抬高的下界）、概率校准，
# 以及模型接近笃定（one-hot）时梯度趋零、深一点的任务收敛变慢。
import torch
import torch.nn.functional as F

torch.manual_seed(0)
logits  = torch.randn(4, 5)
targets = torch.tensor([2, 0, 4, 1])

# ✅ 正确：直接传 raw logits
loss_correct = F.cross_entropy(logits, targets)

# ❌ 错误：手动 softmax 之后再传给 cross_entropy
probs        = F.softmax(logits, dim=-1)
loss_wrong   = F.cross_entropy(probs, targets)         # 不报错！但语义错了

print(f'传 raw logits（正确）  loss = {loss_correct.item():.4f}')
print(f'传 softmax(logits)（错） loss = {loss_wrong.item():.4f}')
print('\n注意错的版本 loss 反而更小——这不代表它更好！它已经不是 NLL，')
print('两个数不可比：概率被再压一次后整体被挤向均匀，loss 有一个抬高的下界')
print('（K=5 时约 0.905，永远到不了 0），模型再笃定 loss 也下不去、梯度跟着消失。')
print('最阴险的是：准确率照常上升、曲线一切正常，肉眼盯训练曲线根本抓不到这个 bug。')
print('所以模型 forward 末尾「永远输出 raw logits」是铁律。')

In [ ]:
# ============================================================
# Cell 8: KL 散度 D_KL(p ∥ q) —— 手算与 PyTorch 实现对照
# ============================================================
# D_KL(p ∥ q) = ∑ p_i log(p_i / q_i)  ，≥ 0，且不对称。
import torch
import torch.nn.functional as F

p = torch.tensor([0.5, 0.5])
q = torch.tensor([0.9, 0.1])

# 手写按定义算
def kl(p, q, eps=1e-12):
    p = torch.as_tensor(p, dtype=torch.float64)
    q = torch.as_tensor(q, dtype=torch.float64)
    return (p * torch.log((p + eps) / (q + eps))).sum().item()

print(f'D_KL(p ∥ q) = {kl(p, q):.4f}')
print(f'D_KL(q ∥ p) = {kl(q, p):.4f}')
print('显然 D_KL(p∥q) ≠ D_KL(q∥p)  —— KL 是不对称的。')

# PyTorch 内置：F.kl_div(input=log q, target=p, reduction='sum')
# 注意 input 是 log q（不是 q！），target 是 p；约定相对怪，但官方就是这样。
log_q = torch.log(q)
kl_pq_torch = F.kl_div(log_q, p, reduction='sum')
print(f'\n用 F.kl_div 算  D_KL(p ∥ q) = {kl_pq_torch.item():.4f}   ← 与手写一致')
print('要点：F.kl_div 第一个参数是 log q（习惯让人困惑），target 是 p。')


In [ ]:
# ============================================================
# Cell 9: KL 不对称性的可视化 —— forward KL vs reverse KL 倾向不同
# ============================================================
# 把 q 用一个一维参数 a 描述：q = (a, 1-a)，a ∈ (0, 1)
# 固定真实分布 p = (0.5, 0.5)，画两种 KL 随 a 变化的曲线，看最优 q 落在哪。
import torch
import numpy as np
import matplotlib.pyplot as plt

p = torch.tensor([0.5, 0.5], dtype=torch.float64)
a_vals = np.linspace(0.01, 0.99, 200)

fwd_kl = []   # forward KL: D_KL(p ∥ q)，q 用 a 参数化
rev_kl = []   # reverse KL: D_KL(q ∥ p)
for a in a_vals:
    q = torch.tensor([a, 1 - a], dtype=torch.float64)
    fwd_kl.append((p * torch.log(p / q)).sum().item())
    rev_kl.append((q * torch.log(q / p)).sum().item())

plt.figure(figsize=(7, 4))
plt.plot(a_vals, fwd_kl, label='forward KL  D(p ∥ q)')
plt.plot(a_vals, rev_kl, label='reverse KL  D(q ∥ p)')
plt.axvline(0.5, color='gray', linestyle='--', linewidth=1)
plt.xlabel('q = (a, 1-a), value of a')
plt.ylabel('KL divergence (nat)')
plt.title('p = (0.5, 0.5) fixed; shape of two KLs as q varies')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('两条曲线都在 a=0.5 取最小值 0 —— 此时 q=p，符合 KL≥0 且只在相等时取 0。')
print('但形状不同：forward KL 在端点 a→0 / a→1 急剧爆发——不允许 q 在 p>0 处趋近 0（零避免 / 质量覆盖）；')
print('             reverse KL 端点饱和较平——容许 q 抛弃 p 的部分区域（零强制）。')
print('（mode-seeking / mode-covering 是多峰分布下的说法，此二元单峰 toy 只演示不对称性本身，详见正文 P03 第 6.3 节注意框。）')


In [ ]:
# ============================================================
# Cell 10: 综合演示 —— 用 cross-entropy 训练一个 5 类小分类器
# ============================================================
# 把 P02 的训练循环骨架沿用过来，验证 "训练 = 最小化 cross-entropy = 等价 MLE"。
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# 玩具数据：5 个类的二维高斯混合，每类 100 个点
N_per_class = 100
K = 5
centers = torch.tensor([[0, 0], [3, 0], [0, 3], [3, 3], [1.5, 1.5]], dtype=torch.float32)
X = torch.cat([centers[c] + 0.5 * torch.randn(N_per_class, 2) for c in range(K)], dim=0)
y = torch.cat([torch.full((N_per_class,), c, dtype=torch.long) for c in range(K)], dim=0)

# 模型：2 → 32 → K  的 MLP，输出 raw logits
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, K),                                   # 注意：不在末尾加 softmax！
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

losses, entropies = [], []
for epoch in range(200):
    logits = model(X)                                   # (N, K)
    loss = F.cross_entropy(logits, y)                   # 等价 MLE / 最小化 NLL

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    # 顺手记录平均预测熵 —— 训练初期 ~ log K，训练后期模型变笃定，熵下降
    with torch.no_grad():
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        ent = -(probs * log_probs).sum(dim=-1).mean().item()
        entropies.append(ent)

import matplotlib.pyplot as plt
import math
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(losses); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('cross-entropy loss')
axes[0].set_title('Training loss = average NLL')
axes[1].plot(entropies)
axes[1].axhline(math.log(K), color='gray', linestyle='--', label=f'log K = {math.log(K):.2f}')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('avg predictive entropy (nat)')
axes[1].set_title('Average entropy of predicted distribution')
axes[1].legend()
plt.tight_layout()
plt.show()

print('训练后期 cross-entropy 接近 0 —— 模型对每个样本几乎都把"真实类别"概率压到 1。')
print('对应地，预测分布的平均熵从初始 ≈ log K 一路下降——分布从均匀变得尖锐。')
print('\n这一切就是 P03 这章讲的：')
print('  最大化 log-likelihood ⇔ 最小化 NLL ⇔ 最小化 cross-entropy ⇔ 让模型分布逼近真实分布')
